# Inspecting `subsample_embeddings.parquet`

Before building the FAISS index for dense retrieval, we open the precomputed Jina V3 chunk embeddings and look at what they actually contain on disk. This notebook is a reference: open it any time later to remember the file's structure without re-deriving it from code.

**What we look at:**
1. How many documents are in the file, and how many chunks does each one have?
2. What is the shape and dtype of a single chunk vector?
3. What are the L2 norms of the vectors? (This tells us whether we need to normalize them ourselves before indexing.)
4. What is the maximum number of chunks per document? (Used later to size search-time oversampling.)

In [ ]:
import sys
from pathlib import Path

# Make `ows_retrieval` importable when running the notebook from notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
from ows_retrieval.data import load_doc_embeddings

## 1. Overall shape

The parquet stores one row per document. Let's see how many rows there are and which columns are present.

In [ ]:
emb = load_doc_embeddings()
print("rows (documents):", len(emb))
print("columns:", emb.columns.tolist())
emb.head(3)

## 2. One row, in detail

The `embeddings` cell for a single document holds a **list of vectors** — one per chunk of that document. Different documents have different numbers of chunks; vectors are all the same length (set by the encoder).

In [ ]:
row0 = emb.iloc[0]
chunks = row0["embeddings"]

print("doc id:", row0["id"])
print("chunk_offsets (char ranges into the source text):")
print(row0["chunk_offsets"])
print()
print("type of embeddings cell:", type(chunks).__name__)
print("number of chunks in this doc:", len(chunks))
print("length of one chunk vector:", len(chunks[0]))
print("dtype of vector entries:", chunks[0].dtype if hasattr(chunks[0], 'dtype') else type(chunks[0][0]).__name__)
print("first 5 numbers of first chunk:", list(chunks[0])[:5])

**What we see:** vectors are 1024-dimensional (Jina V3's output size) and stored in `float16`. FAISS expects `float32`, so we'll cast when flattening.

`chunk_offsets` is character-level start/end positions into the source text — useful only if we later want to highlight *which chunk* matched a query. Not needed for retrieval.

## 3. Chunks-per-document distribution

How many rows will the FAISS index have if we put each chunk on its own row? `sum(chunks_per_doc)`. The **max** also tells us how aggressively we have to oversample at search time: if we want top-k documents, asking FAISS for `k * max_chunks_per_doc` chunks guarantees k distinct docs after the max-sim collapse.

In [ ]:
counts = emb["embeddings"].apply(len)
print("min chunks per doc:    ", counts.min())
print("max chunks per doc:    ", counts.max())
print("mean chunks per doc:   ", round(counts.mean(), 2))
print("median chunks per doc: ", int(counts.median()))
print("total chunks (FAISS rows in chunk-level mode):", int(counts.sum()))
print()
print("distribution:")
print(counts.value_counts().sort_index())

## 4. What are the L2 norms of the vectors?

We use **inner product** as the similarity score (`IndexFlatIP`). On unit-length (L2-normalized) vectors, inner product equals cosine similarity — a pure direction comparison. On un-normalized vectors, vector *length* also affects the score, which is noise we don't want.

Below we sample some chunk vectors and print their L2 norms. If they are all close to 1.0, the vectors are already normalized. If not, we will need to normalize them ourselves before indexing.

In [ ]:
# Sample a few chunk vectors from the first doc and from later docs
sample_vectors = []
for i in [0, 100, 1000, 10000, 30000]:
    for v in emb.iloc[i]["embeddings"]:
        sample_vectors.append(np.asarray(v, dtype=np.float32))
sample = np.stack(sample_vectors)
norms = np.linalg.norm(sample, axis=1)
print("sample size:", len(norms))
print("L2 norms - min: {:.3f}, max: {:.3f}, mean: {:.3f}".format(norms.min(), norms.max(), norms.mean()))

**What we see:** the norms are well above 1, so the vectors are **not** normalized. We'll L2-normalize each chunk vector in `flatten_chunks` before adding it to the FAISS index. After that, every search becomes pure cosine similarity.

## Summary — what this notebook showed us about the data

| Property | Value | Consequence for the dense pipeline |
|---|---|---|
| documents | 60,577 | one row per doc in the parquet |
| chunks total | ~186,473 | number of rows the FAISS index will hold (chunk-level mode) |
| chunk vector dim | 1024 | FAISS `IndexFlatIP(1024)` |
| chunk vector dtype | float16 | cast to float32 before `index.add` |
| max chunks per doc | 5 | request `k * 5` chunks at search time to guarantee k unique docs |
| pre-normalized? | no | L2-normalize at index time, then inner product = cosine similarity |